# 🚀 Модуль 7. Урок 3. Оценка метрик: z-тест для пропорций и t-тест для средних значений

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_03_evaluating_metrics_proportions_means.ipynb)

## 📌 Бизнес-контекст и методология
Для качественной оценки различий:
* **Z-тест для долей (Proportions)**: конверсии (CR, CTR, Retention).
* **T-тест Стьюдента / Уэлча для средних (Means)**: средний чек (AOV), ARPU, время на сайте.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np
from scipy import stats

# Конверсия в группе A и B
conv_a, n_a = 520, 10_000   # CR = 5.2%
conv_b, n_b = 610, 10_000   # CR = 6.1%

p1 = conv_a / n_a
p2 = conv_b / n_b
p_pooled = (conv_a + conv_b) / (n_a + n_b)
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_a + 1/n_b))
z_stat = (p2 - p1) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f"CR Control:  {p1:.2%}")
print(f"CR Variant:  {p2:.2%}")
print(f"Z-статистика: {z_stat:.3f}")
print(f"p-value:      {p_value:.4f}")


CR Control:  5.20%
CR Variant:  6.10%
Z-статистика: 2.756
p-value:      0.0058


### 🎯 Практическое задание 1: Доверительный интервал разницы конверсий
Рассчитайте 95% доверительный интервал для абсолютной разницы конверсий $(p_2 - p_1)$.

In [3]:
### ВАШ КОД ЗДЕСЬ
diff = p2 - p1
se_diff = np.sqrt((p1*(1-p1)/n_a) + (p2*(1-p2)/n_b))
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"95% ДИ абсолютного прироста: [{ci_lower:.4f}, {ci_upper:.4f}]")
assert ci_lower > 0, "Нижняя граница должна быть выше 0 при значимом эффекте"
print("✅ Тест успешно пройден!")


95% ДИ абсолютного прироста: [0.0026, 0.0154]
✅ Тест успешно пройден!


### 💡 t-тест Уэлча vs классический t-тест
Всегда используйте `stats.ttest_ind(..., equal_var=False)` (тест Уэлча). Он устойчив к неравенству дисперсий в группах.